# Defi EquiAlgo - analyse par tranches

Trois decoupages : **region** (5 regions), **cote R** (7 tranches) et **revenu familial** (quintiles
calcules sur `train`). Les tranches de cote R et de revenu sont croisees avec le groupe
(Centre / Eloignee).

- **Partie A, entrainement** : comment la cible corrigee (`cible.cible_individuelle`) differe de
  `decision_octroi` (y_true) dans chaque tranche.
- **Partie B, test** : metriques par tranche du RF de production et du modele corrige, contre la
  reference de merite retenue (`cible.score_merite`, voir `model_corrige.ipynb` §1.5).

Meme decoupage 70/30 que la baseline (stratifie, `random_state=42`).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from analyse import tableau_tranches
from cible import (VARIABLES_MODELE, ajouter_eloignee, ajuster_modele_decision, cible_individuelle,
                   entrainer_modele_cible, octroi_top_k, score_merite)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

demandes = ajouter_eloignee(pd.read_csv('data/donnees_demandes.csv'))
demandes['groupe'] = np.where(demandes['eloignee'] == 1, 'Eloignee', 'Centre')
train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)

# Cible corrigee (entrainement) et reference de merite (test), a partir du modele de decision de train.
modele_valid = ajuster_modele_decision(train)
cible_tr = cible_individuelle(modele_valid, train)
y_ref = octroi_top_k(score_merite(modele_valid, test, reference=train))

# Modele de production (RF de la baseline) et modele corrige, ajustes sur train.
CATEGORIELLES_RF = ['programme_etudes', 'region_administrative', 'code_postal_3']
retirer = ['id_candidat', 'decision_octroi', 'eloignee', 'groupe']
X_tr = pd.get_dummies(train.drop(columns=retirer), columns=CATEGORIELLES_RF)
X_te = pd.get_dummies(test.drop(columns=retirer), columns=CATEGORIELLES_RF).reindex(columns=X_tr.columns, fill_value=0)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
decisions = {
    'RF de production': rf.fit(X_tr, train['decision_octroi']).predict(X_te),
    'modele corrige': octroi_top_k(entrainer_modele_cible(train, cible_tr).predict_proba(test[VARIABLES_MODELE])[:, 1]),
}

# Palette de reference : Centre bleu, Eloignee orange ; textes en encre neutre.
COULEURS_GROUPE = {'Centre': '#2a78d6', 'Eloignee': '#eb6834'}
ENCRE, ENCRE_SECONDAIRE, GRILLE = '#0b0b0b', '#52514e', '#e4e3df'


def styliser(ax, titre=None):
    """Axes discrets : grille legere, sans cadre haut/droite."""
    if titre:
        ax.set_title(titre, color=ENCRE, fontsize=11, loc='left')
    ax.grid(color=GRILLE, lw=0.8)
    ax.set_axisbelow(True)
    for cote in ['top', 'right']:
        ax.spines[cote].set_visible(False)
    ax.tick_params(colors=ENCRE_SECONDAIRE)
    return ax

## Definition des tranches

Les bornes de revenu sont les quintiles de `train`, appliquees telles quelles a `test` : une tranche
designe la meme plage de revenu dans les deux jeux.

In [ ]:
BORNES_R = [0, 26, 27, 28, 29, 30, 32, 45]
BORNES_REVENU = np.quantile(train['revenu_familial_estime'], [0, 0.2, 0.4, 0.6, 0.8, 1])
BORNES_REVENU[0], BORNES_REVENU[-1] = 0, np.inf
ETIQUETTES_REVENU = [f'Q{i + 1} {a / 1000:.0f}-{b / 1000:.0f}k' if np.isfinite(b) else f'Q{i + 1} {a / 1000:.0f}k+'
                     for i, (a, b) in enumerate(zip(BORNES_REVENU[:-1], BORNES_REVENU[1:]))]


def tranches(df):
    """Les trois decoupages d'un jeu de donnees, chacun pret pour un groupby ou un MetricFrame."""
    groupe = df['groupe'].rename('groupe')
    cote_r = pd.cut(df['cote_r_equivalent'], BORNES_R).astype(str).rename('cote R')
    revenu = pd.cut(df['revenu_familial_estime'], BORNES_REVENU, labels=ETIQUETTES_REVENU).astype(str).rename('revenu')
    return {
        'region': df['region_administrative'].rename('region'),
        'cote R': pd.concat([cote_r, groupe], axis=1),
        'revenu': pd.concat([revenu, groupe], axis=1),
    }


tranches_tr, tranches_te = tranches(train), tranches(test)
print('Quintiles de revenu (train) :', ETIQUETTES_REVENU)

## Partie A - Entrainement : cible corrigee contre y_true

La cible corrigee part de la decision du comite et la corrige dossier par dossier : penalite regionale
retiree, revenu et distance neutralises. Avec p la probabilite d'octroi du comite et q le score de
merite :
- un dossier accorde garde une cible de min(1, q / p) : elle baisse si le revenu l'avait aide ;
- un dossier refuse recoit max(0, (q - p) / (1 - p)) : elle monte s'il avait ete penalise.

Par tranche : taux d'octroi observe (y_true), moyenne de la cible, et ampleur de la correction chez
les refuses et chez les accordes.

In [ ]:
def cible_contre_y_true(df, cible, tranche):
    """Par tranche : effectif, taux y_true, moyenne de la cible et correction recue par les refuses."""
    donnees = pd.DataFrame({'y_true': df['decision_octroi'].to_numpy(), 'cible': np.asarray(cible)}, index=df.index)
    cles = [tranche[c] for c in tranche.columns] if isinstance(tranche, pd.DataFrame) else [tranche]
    par_tranche = donnees.groupby(cles, observed=True)
    refuses = donnees[donnees['y_true'] == 0].groupby([c[donnees['y_true'] == 0] for c in cles], observed=True)['cible']
    accordes = donnees[donnees['y_true'] == 1].groupby([c[donnees['y_true'] == 1] for c in cles], observed=True)['cible']
    return pd.DataFrame({
        'effectif': par_tranche.size(),
        'taux y_true': par_tranche['y_true'].mean(),
        'moyenne cible': par_tranche['cible'].mean(),
        'ecart (cible - y_true)': par_tranche['cible'].mean() - par_tranche['y_true'].mean(),
        'cible moyenne des refuses': refuses.mean(),
        'refuses avec cible > 0.5': refuses.apply(lambda s: (s > 0.5).mean()),
        'cible moyenne des accordes': accordes.mean(),
        'accordes avec cible < 0.5': accordes.apply(lambda s: (s < 0.5).mean()),
    })


cibles_tranches = {nom: cible_contre_y_true(train, cible_tr, t) for nom, t in tranches_tr.items()}
for nom, tableau in cibles_tranches.items():
    print(f'--- {nom} ---')
    print(tableau.round(3), end='\n\n')

In [ ]:
def tracer_cible_contre_y_true(cibles_tranches):
    """Par decoupage : taux y_true (cercle vide) et moyenne de la cible (cercle plein), par groupe."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
    for ax, (nom, tableau) in zip(axes, cibles_tranches.items()):
        if tableau.index.nlevels == 1:
            # Region : une tranche = une region, coloree selon son groupe.
            groupes = train.groupby('region_administrative')['groupe'].first().reindex(tableau.index)
            x = np.arange(len(tableau))
            for g, couleur in COULEURS_GROUPE.items():
                m = (groupes == g).to_numpy()
                ax.scatter(x[m], tableau['taux y_true'][m], s=70, facecolor='white', edgecolor=couleur, lw=2, zorder=3)
                ax.scatter(x[m], tableau['moyenne cible'][m], s=70, color=couleur, edgecolor='white', lw=1.5, zorder=4)
                ax.vlines(x[m], tableau['taux y_true'][m], tableau['moyenne cible'][m], color=couleur, lw=1.5)
            ax.set_xticks(x, [r.replace('-', '-\n', 1) for r in tableau.index], fontsize=8)
        else:
            niveaux = tableau.index.get_level_values(0).unique()
            x = np.arange(len(niveaux))
            for g, couleur in COULEURS_GROUPE.items():
                t = tableau.xs(g, level='groupe').reindex(niveaux)
                ax.plot(x, t['taux y_true'], '--o', color=couleur, lw=1.5, ms=7, mfc='white', mew=2)
                ax.plot(x, t['moyenne cible'], '-o', color=couleur, lw=2, ms=7, mec='white', mew=1.5)
            ax.set_xticks(x, niveaux, fontsize=8, rotation=30 if nom == 'revenu' else 0)
        styliser(ax, f'Par {nom}')
    axes[0].set_ylabel("taux d'octroi", color=ENCRE_SECONDAIRE)
    legende = [plt.Line2D([], [], color=c, lw=2, marker='o', mec='white', label=g) for g, c in COULEURS_GROUPE.items()]
    legende += [plt.Line2D([], [], color=ENCRE_SECONDAIRE, ls='--', marker='o', mfc='white', label='y_true (decision_octroi)'),
                plt.Line2D([], [], color=ENCRE_SECONDAIRE, ls='-', marker='o', label='moyenne de la cible corrigee')]
    fig.legend(handles=legende, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()


tracer_cible_contre_y_true(cibles_tranches)

**Lecture.**
- **Region** : la cible ajoute 18 a 20 points dans chacune des trois regions eloignees (Bas-Saint-Laurent
  de 28 a 46 %, Cote-Nord de 26 a 46 %, Gaspesie de 27 a 47 %), et baisse a peine au Centre (moins d'un
  point). Au Centre, deux corrections se compensent : les accordes aises descendent, les refuses modestes
  montent. Apres correction, l'ecart entre les deux blocs est de 1 a 2.5 points.
- **Cote R** : en region, la correction est la plus forte pres du seuil. Elle ajoute 48 points entre 28
  et 29 (de 25 a 73 %), et 34 a 35 points dans les tranches voisines. Au Centre, la cible reste a moins
  de 3.5 points de y_true dans chaque tranche. A cote R egale, les candidats eloignes passent devant
  (73 % contre 60 % entre 28 et 29) : l'etalon valorise les heures travaillees, plus nombreuses en
  region.
- **Revenu** : la pente de revenu du comite disparait. Au Centre, y_true passe de 41 % (Q1) a 56 % (Q5)
  et la cible reste entre 47 et 49 % ; en region, y_true passe de 19 a 35 % et la cible reste entre 43
  et 49 %. C'est l'effet voulu : l'etalon ne tient pas compte du revenu (sondes HxBuddy).

### Distribution de la cible

Distribution de la cible des refuses et des accordes, par groupe ; puis cible des refuses eloignes par
tranche de cote R, la ou la correction est la plus forte.

In [ ]:
y_true_tr = train['decision_octroi'].to_numpy()
groupe_tr = train['groupe'].to_numpy()
print('Repartition de la cible (dossiers de train)')
print(pd.crosstab([np.where(y_true_tr == 1, 'accorde', 'refuse'), groupe_tr],
                  pd.cut(cible_tr, [-0.001, 0, 0.25, 0.5, 0.75, 0.999, 1],
                         labels=['0', '0-0.25', '0.25-0.5', '0.5-0.75', '0.75-1', '1']),
                  rownames=['y_true', 'groupe'], colnames=['cible']))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
bacs = np.linspace(0, 1, 26)
for ax, (decision, titre) in zip(axes[:2], [(0, 'Cible des refuses (y_true = 0)'), (1, 'Cible des accordes (y_true = 1)')]):
    for g, couleur in COULEURS_GROUPE.items():
        m = (y_true_tr == decision) & (groupe_tr == g)
        ax.hist(cible_tr[m], bins=bacs, histtype='step', lw=2, color=couleur, label=f'{g} ({m.sum()})')
    ax.set_yscale('log')
    ax.set_xlabel('cible corrigee', color=ENCRE_SECONDAIRE)
    styliser(ax, titre)
axes[0].set_ylabel('dossiers (echelle log)', color=ENCRE_SECONDAIRE)
axes[0].legend(frameon=False)

refuses_eloignes = (y_true_tr == 0) & (groupe_tr == 'Eloignee')
cible_refuses = cible_tr[refuses_eloignes]
tranche_r_refuses = tranches_tr['cote R']['cote R'].to_numpy()[refuses_eloignes]
ordre = [t for t in sorted(set(tranche_r_refuses), key=lambda t: float(t[1:].split(',')[0]))
         if (tranche_r_refuses == t).sum() >= 10]
axes[2].boxplot([cible_refuses[tranche_r_refuses == t] for t in ordre], tick_labels=ordre, widths=0.5,
                medianprops={'color': COULEURS_GROUPE['Eloignee'], 'lw': 2},
                boxprops={'color': ENCRE_SECONDAIRE}, whiskerprops={'color': ENCRE_SECONDAIRE},
                capprops={'color': ENCRE_SECONDAIRE}, flierprops={'markersize': 3, 'markeredgecolor': GRILLE})
axes[2].set_xlabel('tranche de cote R (moins de 10 refuses : omise)', color=ENCRE_SECONDAIRE)
styliser(axes[2], 'Cible des refuses eloignes, par cote R')
plt.tight_layout()
plt.show()

**Lecture.**
- **Accordes eloignes** : 750 sur 753 gardent une cible de 1. Le retrait de la penalite regionale
  l'emporte toujours sur la neutralisation du revenu.
- **Accordes du Centre** : 1 046 sur 2 043 voient leur cible baisser, dont 159 sous 0.75. Leur revenu
  moyen est de 105 k$, contre 53 k$ pour ceux qui gardent 1 : ce sont les candidats que leur revenu
  avait aides.
- **Refuses du Centre** : 1 274 sur 2 182 recoivent une cible positive, presque toujours sous 0.25.
  Leur revenu moyen est de 51 k$, contre 100 k$ pour ceux qui restent a 0.
- **Refuses eloignes** : ce sont eux que la correction touche le plus. 24 % passent au-dessus de 0.5,
  et la cible mediane croit avec la cote R : 0.01 sous 26, 0.15 entre 26 et 27, 0.38 entre 27 et 28,
  0.67 entre 28 et 29, 0.82 entre 29 et 30 et 0.90 entre 30 et 32.
- La cible n'atteint jamais 1 pour un refus (maximum de 0.94) : un refus du comite reste une information
  sur le dossier, que l'abduction conserve.

## Partie B - Test : metriques par tranche

Metriques contre la reference de merite, pour les deux modeles. La TPR d'une tranche sans meritant
et la FPR d'une tranche sans non-meritant sont laissees vides. Les tranches de moins de 100 dossiers
sont signalees par `*`.

In [ ]:
def comparer_modeles(tranche):
    """Metriques completes par tranche, RF de production et modele corrige cote a cote."""
    tableau = pd.concat({nom: tableau_tranches(y_ref, d, tranche, complet=True) for nom, d in decisions.items()},
                        axis=1)
    effectif = tableau[('RF de production', 'effectif')]
    tableau = tableau.drop(columns=[('modele corrige', 'effectif'), ('modele corrige', 'meritants')])
    tableau[('', 'faible effectif')] = np.where(effectif < 100, '*', '')
    return tableau


metriques_tranches = {nom: comparer_modeles(t) for nom, t in tranches_te.items()}
for nom, tableau in metriques_tranches.items():
    print(f'--- {nom} ---')
    print(tableau.round(3), end='\n\n')

In [ ]:
def tracer_metriques(metriques_tranches, metriques=('taux_selection', 'tpr'), min_effectif=10):
    """Une ligne par decoupage, une colonne par metrique ; RF en pointille, modele corrige en trait plein."""
    titres = {'taux_selection': 'taux de selection', 'tpr': 'TPR (contre la reference)'}
    fig, axes = plt.subplots(len(metriques_tranches), len(metriques), figsize=(13, 4 * len(metriques_tranches)))
    for ligne, (nom, tableau) in zip(axes, metriques_tranches.items()):
        for ax, metrique in zip(ligne, metriques):
            if tableau.index.nlevels == 1:
                groupes = test.groupby('region_administrative')['groupe'].first().reindex(tableau.index)
                x = np.arange(len(tableau))
                for modele, forme in [('RF de production', 'o'), ('modele corrige', 'D')]:
                    for g, couleur in COULEURS_GROUPE.items():
                        m = (groupes == g).to_numpy()
                        plein = modele == 'modele corrige'
                        ax.scatter(x[m], tableau[(modele, metrique)][m], s=70, marker=forme,
                                   facecolor=couleur if plein else 'white', edgecolor=couleur, lw=2, zorder=3)
                ax.set_xticks(x, [r.replace('-', '-\n', 1) for r in tableau.index], fontsize=8)
            else:
                niveaux = tableau.index.get_level_values(0).unique()
                x = np.arange(len(niveaux))
                for g, couleur in COULEURS_GROUPE.items():
                    t = tableau.xs(g, level='groupe').reindex(niveaux)
                    for modele, style in [('RF de production', '--'), ('modele corrige', '-')]:
                        valeurs = t[(modele, metrique)].where(t[('RF de production', 'meritants')] >= min_effectif) \
                            if metrique == 'tpr' else t[(modele, metrique)]
                        ax.plot(x, valeurs, style + 'o', color=couleur, lw=2, ms=6,
                                mfc='white' if style == '--' else couleur, mec=couleur, mew=1.5)
                ax.set_xticks(x, niveaux, fontsize=8, rotation=30 if nom == 'revenu' else 0)
            ax.set_ylim(-0.03, 1.05)
            styliser(ax, f'{titres[metrique]} par {nom}')
    legende = [plt.Line2D([], [], color=c, lw=2, label=g) for g, c in COULEURS_GROUPE.items()]
    legende += [plt.Line2D([], [], color=ENCRE_SECONDAIRE, ls='--', marker='o', mfc='white', label='RF de production'),
                plt.Line2D([], [], color=ENCRE_SECONDAIRE, ls='-', marker='o', label='modele corrige')]
    fig.legend(handles=legende, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0))
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


tracer_metriques(metriques_tranches)

**Lecture.** La reference de merite et le modele corrige derivent du meme modele de decision : la TPR
d'environ 1 du modele corrige est en partie **circulaire**. L'information se trouve surtout dans les
colonnes du RF de production ; pour un controle independant, voir `audit_rapport.ipynb` §5.

- **Region** : le RF a une TPR de 65 a 73 % dans chaque region eloignee, contre 98 a 100 % au Centre. A
  l'inverse, sa FPR est nulle en region et de 10 a 11 % au Centre (precision de 0.85 a 0.88) : il
  accorde des bourses a des candidats non meritants du Centre, au detriment de meritants eloignes.
  Apres correction, la TPR est d'au moins 99.3 % et la FPR d'au plus 0.3 % partout.
- **Cote R** : entre 28 et 29, le RF accorde la bourse a 70.5 % des dossiers du Centre (FPR de 0.57)
  contre 3.2 % des dossiers eloignes (TPR de 0.05). Le modele corrige accorde a 36.5 % et 66.9 %.
- **Revenu** : la TPR du RF pour les candidats eloignes est de 60 a 74 % dans **tous** les quintiles, et
  la plus basse dans le plus pauvre (60 % en Q1) : les candidats eloignes et modestes cumulent les deux
  biais du comite. Au Centre, la FPR du RF augmente avec le revenu (de 4 % en Q1 a 15 % en Q5) : il
  accorde des bourses a des candidats aises non meritants. Apres correction, la TPR est d'au moins
  98.7 % partout, et les taux de selection des deux groupes restent entre 37 et 43 % dans chaque
  quintile.